# Topic Visualization


In [1]:
import mailbox
import numpy as np
from pathlib import Path

from bertopic import BERTopic
from usenet_no.mbox_utils import message_factory, get_message_body

MODEL = "codefuse-ai/F2LLM-v2-0.6B"
# MODEL = "NbAiLab/nb-sbert-v2-large"
SELECTION = [
    "no.religion",
    "no.bil",
    "no.musikk",
    "no.slekt",
    "no.litteratur",
    "no.prat.politikk",
]
NR_TOPICS = 10  # set to an int to match --nr-topics in topic_modelling.py
# NR_TOPICS = None

RUN_TAG = "_".join(sorted(SELECTION))
umap_cache = Path(f"../data/embeddings/umap_embeddings/{MODEL}/{RUN_TAG}.npy")

if NR_TOPICS is not None:
    RUN_TAG += f"_nr{NR_TOPICS}"

embedding_dir = Path(f"../data/embeddings/{MODEL}")
topics_dir = Path(f"../data/topics/{MODEL}/{RUN_TAG}")
source_dirs = {
    "ia": Path("../data/internet_archive/date_filtered"),
    "nwa": Path("../data/nwa_90s/utf_8_data"),
}

all_embeddings = []
embedding_indexer = []
text_indexer = []

for f in sorted(embedding_dir.iterdir()):
    if f.stem.endswith("_index"):
        continue

    mbox_stem, source = f.stem.rsplit("_", 1)
    if mbox_stem not in SELECTION:
        continue

    embs = np.load(f)
    mbox_file = source_dirs[source] / f"{mbox_stem}.mbox"
    messages = list(mailbox.mbox(str(mbox_file), factory=message_factory))

    index_file = embedding_dir / f"{f.stem}_index.npy"
    if index_file.exists():
        indices = np.load(index_file)
        bodies = [get_message_body(messages[i]) for i in indices]
    else:
        bodies = [get_message_body(m) for m in messages]

    all_embeddings.extend(embs)
    embedding_indexer += [f.stem] * len(embs)
    text_indexer += bodies

all_embeddings = np.array(all_embeddings)
print(f"Loaded {len(text_indexer)} documents, embeddings shape: {all_embeddings.shape}")

Loaded 146997 documents, embeddings shape: (146997, 1024)


In [2]:
umap_2d = np.load(umap_cache)

topic_model = BERTopic.load(str(topics_dir / "bertopic_model"))
topics, _ = topic_model.transform(text_indexer, all_embeddings)
topics = np.array(topics)

topic_info = topic_model.get_topic_info().set_index("Topic")

print(f"UMAP shape: {umap_2d.shape}")
print("Unique topics:")
for t in sorted(set(topics)):
    if t == -1:
        print(f"  -1: outliers: {topic_info.loc[t, 'Representation']}")
    else:
        words = ", ".join(topic_info.loc[t, "Representation"])
        print(f"TOPIC NUMBER {t}: REPRESENTATIVE WORDS: {words}")
        # print("\nREPRESENTATIVE MESSAGES:")
        # print(
        #     "NEXT REPRESENTATIVE MESSAGE:\n".join(
        #         topic_info.loc[t, "Representative_Docs"]
        #     ),
        #     "\n\n",
        # )

UMAP shape: (146997, 2)
Unique topics:
  -1: outliers: ['det', 'er', 'og', 'som', 'jeg', 'ikke', 'at', 'en', 'har', 'til']
TOPIC NUMBER 0: REPRESENTATIVE WORDS: det, er, og, som, jeg, ikke, at, en, du, har
TOPIC NUMBER 1: REPRESENTATIVE WORDS: the, of, to, and, that, is, in, not, you, are
TOPIC NUMBER 2: REPRESENTATIVE WORDS: the, you, and, to, your, this, of, it, is, in
TOPIC NUMBER 3: REPRESENTATIVE WORDS: er, det, og, som, test, jeg, en, har, at, til
TOPIC NUMBER 4: REPRESENTATIVE WORDS: the, of, and, 13107, to, af, 24, that, og, in
TOPIC NUMBER 5: REPRESENTATIVE WORDS: er, det, til, og, jeg, en, som, har, med, av
TOPIC NUMBER 6: REPRESENTATIVE WORDS: hebrew, the, tanakh, and, book, plates, volume, printing, been, of
TOPIC NUMBER 7: REPRESENTATIVE WORDS: er, det, at, ikke, som, jeg, og, den, du, en
TOPIC NUMBER 8: REPRESENTATIVE WORDS: odd, gif, grafikkfiler, jpg, eudora, program, netscape, agent, sorensen, trengs


In [ ]:
import colorsys
import plotly.graph_objects as go


def hsl_to_hex(h, s, lightness):
    r, g, b = colorsys.hls_to_rgb(h / 360, lightness / 100, s / 100)
    return f"#{int(r * 255):02x}{int(g * 255):02x}{int(b * 255):02x}"


def topic_label(t, n_words=5):
    if t == -1:
        return "outliers (-1)"
    words = ", ".join(topic_info.loc[t, "Representation"][:n_words])
    return f"Topic {t}: {words}"


topic_info = topic_model.get_topic_info().set_index("Topic")
unique_topics = sorted(set(topics))
color_map = {
    t: "lightgrey"
    if t == -1
    else hsl_to_hex(int(i * 360 / (len(unique_topics) - 1)), 70, 50)
    for i, t in enumerate(unique_topics)
}

hover_texts = np.array(
    [
        f"<b>{topic_label(t)}</b><br><i>{stem.rsplit('_', 1)[0]}</i><br>"
        + body[:400].replace("\n", "<br>")
        for t, stem, body in zip(topics, embedding_indexer, text_indexer)
    ]
)

fig = go.Figure()

for t in unique_topics:
    mask = topics == t
    fig.add_trace(
        go.Scattergl(
            x=umap_2d[mask, 0],
            y=umap_2d[mask, 1],
            mode="markers",
            marker=dict(size=4, color=color_map[t], opacity=0.5 if t == -1 else 0.7),
            name=topic_label(t, n_words=1),
            text=hover_texts[mask],
            hovertemplate="%{text}<extra></extra>",
        )
    )

fig.update_layout(
    title="Norwegian Usenet message embeddings (color=BERTopic topic)",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    width=1100,
    height=750,
    legend=dict(font=dict(size=9)),
)
fig.show()